# Eval v2: Turn Detector with Threshold Classification
- P(im_end) > 0.5 → positive (turn complete)
- P(im_end) < 0.5 → negative (turn incomplete)

In [1]:
import torch
import math
import json
import numpy as np
import torch.nn.functional as F
from chinidataset import StreamingDataset
from transformers import AutoTokenizer, Qwen3ForCausalLM
from liger_kernel.transformers import apply_liger_kernel_to_qwen3, LigerFusedLinearCrossEntropyLoss
from tqdm import tqdm
import glob
import os
import pandas as pd

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Scicom-intl/turn-detector-Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16).cuda().eval()

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

## Config

In [5]:
TEST_FILE = './parquet-test-synthetic'
THRESHOLD = 0.5

## Load Model

In [8]:
apply_liger_kernel_to_qwen3(
    rope=True,
    swiglu=True,
    rms_norm=True,
    cross_entropy=False,
    fused_linear_cross_entropy=False,
)

class Model(Qwen3ForCausalLM):
    def __init__(self, config):
        super().__init__(config)
        self.loss = LigerFusedLinearCrossEntropyLoss(reduction='sum')

    def forward(self, input_ids, attention_mask=None, position_ids=None,
                labels=None, **kwargs):
        super_out = self.model.forward(
            input_ids=input_ids,
            position_ids=position_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            **kwargs,
        )
        if labels is not None:
            embeddings = super_out.last_hidden_state
            embeddings = embeddings[:, :-1].reshape(-1, embeddings.shape[-1])
            labels = labels[..., 1:].contiguous().reshape(-1)
            loss = self.loss(self.lm_head.weight, embeddings, labels)
            loss = loss / (labels != -100).sum()
            return {'loss': loss, 'hidden_states': super_out.last_hidden_state}
        return super_out

model = Model.from_pretrained(
    model_id,
    attn_implementation='flash_attention_2',
    torch_dtype='bfloat16',
)
model.eval()
model.cuda()

Model(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024, padding_idx=151643)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): LigerRMSNorm((128,), eps=1e-06, offset=0.0, in_place=True, row_mode=None)
          (k_norm): LigerRMSNorm((128,), eps=1e-06, offset=0.0, in_place=True, row_mode=None)
        )
        (mlp): LigerSwiGLUMLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        )
        (input_layernorm): 

In [26]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
IM_END_TOKEN_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
print(f'<|im_end|> token ID: {IM_END_TOKEN_ID}')

<|im_end|> token ID: 151645


## P(im_end) function

In [27]:
def get_im_end_prob(text):
    """Get probability that the turn should end.
    Strips trailing <|im_end|> so the model predicts whether to emit it.
    """
    ix = text.rfind('<|im_end|>')
    if ix != -1 and ix == len(text) - len('<|im_end|>'):
        text = text[:ix]
    
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        logits = Qwen3ForCausalLM.forward(model, **inputs).logits
    logprob = F.log_softmax(logits[0, -1], dim=-1)[IM_END_TOKEN_ID].item()
    prob = math.exp(logprob)
    return prob, logprob

## Run on test set

In [28]:
ds = StreamingDataset(local=TEST_FILE)
print(f'Test samples: {len(ds)}')

results = []
for i in tqdm(range(len(ds))):
    row = ds[i]
    meta = json.loads(row['text'])
    true_label = meta['label']
    language = meta.get('language', 'unknown')
    
    # Decode tokens to text
    text = tokenizer.decode(row['input_ids'].astype(np.int64), skip_special_tokens=False)
    
    prob, logprob = get_im_end_prob(text)
    pred_label = 'positive' if prob > THRESHOLD else 'negative'
    correct = pred_label == true_label
    
    results.append({
        'idx': i,
        'true_label': true_label,
        'pred_label': pred_label,
        'correct': correct,
        'prob': prob,
        'logprob': logprob,
        'language': language,
        'text_preview': text[:200],
    })

df = pd.DataFrame(results)
print(f'\nDone: {len(df)} samples evaluated')

Test samples: 1200


100%|██████████| 1200/1200 [00:43<00:00, 27.43it/s]


Done: 1200 samples evaluated


## Overall accuracy

In [29]:
accuracy = df['correct'].mean()
print(f'Threshold: {THRESHOLD}')
print(f'Overall accuracy: {accuracy:.4f} ({df["correct"].sum()}/{len(df)})')

# Per class
for label in ['positive', 'negative']:
    subset = df[df['true_label'] == label]
    acc = subset['correct'].mean()
    print(f'  {label}: {acc:.4f} ({subset["correct"].sum()}/{len(subset)})')

Threshold: 0.5
Overall accuracy: 0.9700 (1164/1200)
  positive: 0.9400 (564/600)
  negative: 1.0000 (600/600)


## Per language accuracy

In [30]:
print(f'Threshold: {THRESHOLD}\n')
for lang in sorted(df['language'].unique()):
    subset = df[df['language'] == lang]
    acc = subset['correct'].mean()
    pos = subset[subset['true_label'] == 'positive']
    neg = subset[subset['true_label'] == 'negative']
    pos_acc = pos['correct'].mean() if len(pos) > 0 else 0
    neg_acc = neg['correct'].mean() if len(neg) > 0 else 0
    print(f'{lang}:')
    print(f'  overall:  {acc:.4f} ({subset["correct"].sum()}/{len(subset)})')
    print(f'  positive: {pos_acc:.4f} ({pos["correct"].sum()}/{len(pos)})')
    print(f'  negative: {neg_acc:.4f} ({neg["correct"].sum()}/{len(neg)})')

Threshold: 0.5

chinese-english:
  overall:  0.9700 (97/100)
  positive: 0.9400 (47/50)
  negative: 1.0000 (50/50)
chinese-malay:
  overall:  0.9700 (97/100)
  positive: 0.9400 (47/50)
  negative: 1.0000 (50/50)
chinese-tamil:
  overall:  0.9700 (97/100)
  positive: 0.9400 (47/50)
  negative: 1.0000 (50/50)
english-chinese:
  overall:  1.0000 (100/100)
  positive: 1.0000 (50/50)
  negative: 1.0000 (50/50)
english-malay:
  overall:  0.9700 (97/100)
  positive: 0.9400 (47/50)
  negative: 1.0000 (50/50)
english-tamil:
  overall:  0.9400 (94/100)
  positive: 0.8800 (44/50)
  negative: 1.0000 (50/50)
malay-chinese:
  overall:  0.9700 (97/100)
  positive: 0.9400 (47/50)
  negative: 1.0000 (50/50)
malay-english:
  overall:  0.9300 (93/100)
  positive: 0.8600 (43/50)
  negative: 1.0000 (50/50)
malay-tamil:
  overall:  0.9500 (95/100)
  positive: 0.9000 (45/50)
  negative: 1.0000 (50/50)
tamil-chinese:
  overall:  1.0000 (100/100)
  positive: 1.0000 (50/50)
  negative: 1.0000 (50/50)
tamil-engl

## Confusion matrix

In [31]:
tp = len(df[(df['true_label'] == 'positive') & (df['pred_label'] == 'positive')])
tn = len(df[(df['true_label'] == 'negative') & (df['pred_label'] == 'negative')])
fp = len(df[(df['true_label'] == 'negative') & (df['pred_label'] == 'positive')])
fn = len(df[(df['true_label'] == 'positive') & (df['pred_label'] == 'negative')])

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f'Threshold: {THRESHOLD}')
print(f'\nConfusion Matrix:')
print(f'                 Predicted')
print(f'                 Pos    Neg')
print(f'  Actual Pos     {tp:<6} {fn}')
print(f'  Actual Neg     {fp:<6} {tn}')
print(f'\nPrecision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1:        {f1:.4f}')

Threshold: 0.5

Confusion Matrix:
                 Predicted
                 Pos    Neg
  Actual Pos     564    36
  Actual Neg     0      600

Precision: 1.0000
Recall:    0.9400
F1:        0.9691


## Probability distribution

In [32]:
print('Positive samples (should be > 0.5):')
pos_probs = df[df['true_label'] == 'positive']['prob']
print(f'  mean={pos_probs.mean():.4f}  median={pos_probs.median():.4f}  min={pos_probs.min():.4f}  max={pos_probs.max():.4f}')

print('\nNegative samples (should be < 0.5):')
neg_probs = df[df['true_label'] == 'negative']['prob']
print(f'  mean={neg_probs.mean():.4f}  median={neg_probs.median():.4f}  min={neg_probs.min():.4f}  max={neg_probs.max():.4f}')

Positive samples (should be > 0.5):
  mean=0.8817  median=0.9569  min=0.0046  max=0.9997

Negative samples (should be < 0.5):
  mean=0.0010  median=0.0000  min=0.0000  max=0.2509


## Threshold sweep

In [33]:
print(f'{"Threshold":<12} {"Accuracy":<10} {"Precision":<10} {"Recall":<10} {"F1":<10}')
print('-' * 52)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    df['_pred'] = df['prob'].apply(lambda x: 'positive' if x > t else 'negative')
    acc = (df['_pred'] == df['true_label']).mean()
    _tp = len(df[(df['true_label'] == 'positive') & (df['_pred'] == 'positive')])
    _fp = len(df[(df['true_label'] == 'negative') & (df['_pred'] == 'positive')])
    _fn = len(df[(df['true_label'] == 'positive') & (df['_pred'] == 'negative')])
    _prec = _tp / (_tp + _fp) if (_tp + _fp) > 0 else 0
    _rec = _tp / (_tp + _fn) if (_tp + _fn) > 0 else 0
    _f1 = 2 * _prec * _rec / (_prec + _rec) if (_prec + _rec) > 0 else 0
    marker = ' <<<' if t == THRESHOLD else ''
    print(f'{t:<12} {acc:<10.4f} {_prec:<10.4f} {_rec:<10.4f} {_f1:<10.4f}{marker}')
df.drop('_pred', axis=1, inplace=True)

Threshold    Accuracy   Precision  Recall     F1        
----------------------------------------------------
0.1          0.9933     0.9966     0.9900     0.9933    
0.2          0.9900     0.9966     0.9833     0.9899    
0.3          0.9883     1.0000     0.9767     0.9882    
0.4          0.9850     1.0000     0.9700     0.9848    
0.5          0.9700     1.0000     0.9400     0.9691     <<<
0.6          0.9608     1.0000     0.9217     0.9592    
0.7          0.9467     1.0000     0.8933     0.9437    
0.8          0.9092     1.0000     0.8183     0.9001    
0.9          0.8392     1.0000     0.6783     0.8083    


## Worst predictions

In [34]:
wrong = df[~df['correct']].sort_values('prob', ascending=False)
print(f'Wrong predictions: {len(wrong)}/{len(df)}')
print()
for _, row in wrong.head(10).iterrows():
    print(f'[{row["true_label"]}→{row["pred_label"]}] prob={row["prob"]:.4f} lang={row["language"]}')
    print(f'  {row["text_preview"][:150]}')
    print()

Wrong predictions: 36/1200

[positive→negative] prob=0.4970 lang=malay-english
  <|im_start|>assistance
Hi, saya nak update alamat rumah dengan nombor telefon untuk polisi insurans saya. Boleh tolong?<|im_end|>
<|im_start|>assistan

[positive→negative] prob=0.4950 lang=chinese-tamil
  <|im_start|>assistance
你好，我想问一下，如果我要汇钱去新加坡，海外汇款的收款人要怎么新增？我在app里面找不到。<|im_end|>
<|im_start|>assistance
您好，这里为您说明一下。要新增新加坡的海外收款人，您可以按照以下步骤操作：

1. 登录本行的m

[positive→negative] prob=0.4931 lang=tamil-english
  <|im_start|>assistance
வணக்கம் அண்ணே, நான் இப்போ prepaid ல தான் use பண்ணிக்கிறேன். Postpaidக்கு மாற்றணும் என்று நினைக்கிறேன். Process எப்படி இருக்கும்

[positive→negative] prob=0.4912 lang=tamil-malay
  <|im_start|>assistance
ஹலோ, நான் நேற்று உங்கள் websiteல ticket book பண்ணிருக்கேன். இப்போ அதுக்கு travel insurance add பண்ணணும். எப்படி செய்யலாம்?<|im_

[positive→negative] prob=0.4874 lang=malay-english
  <|im_start|>assistance
Hi, saya nak tanya pasal perkhidmatan untuk kanak-kanak bawah umur yang naik k

## Manual test

In [35]:
test_prompts = [
    # Complete - should be > 0.5
    '<|im_start|>user\nHello, saya nak tanya pasal bil saya.<|im_end|>\n<|im_start|>assistant\nBoleh, sila berikan nombor akaun anda.',
    # Incomplete - should be < 0.5
    '<|im_start|>user\nHello, saya nak tanya pasal bil saya.<|im_end|>\n<|im_start|>assistant\nBoleh, sila berikan nombor',
    # Complete
    '<|im_start|>user\nTerima kasih banyak atas bantuan anda.',
    # Incomplete
    '<|im_start|>user\nSaya nak tanya pasal',
]

for prompt in test_prompts:
    prob, lp = get_im_end_prob(prompt)
    label = 'positive' if prob > THRESHOLD else 'negative'
    print(f'[{label:>8}] prob={prob:.4f}')

[positive] prob=0.7359
[negative] prob=0.0000
[negative] prob=0.0000
[negative] prob=0.0000
